# 08 — Evaluate the improved local-patch residual model

This notebook evaluates the experimental model from notebook 07.

It compares:

- frozen DINOv2 CLS prototype;
- frozen DINOv2 mean-patch prototype;
- frozen **local patch matching** with no GNN;
- improved residual GATv2 using DINO-preserving node residuals and local patch matching.

By default this notebook evaluates on the **validation split**. Keep it on validation while deciding whether the architectural changes are useful. Only switch to the test split after the design and hyperparameters are frozen.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from cross_image_glot.config import DEFAULT_PATHS
from cross_image_glot.storage import restore_feature_splits
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder
from cross_image_glot.baselines import frozen_baseline_episode
from cross_image_glot.models import (
    PatchGATv2Encoder,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)
from cross_image_glot.training import evaluate_residual_feature_episode

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Experimental model definitions

These are repeated inline so the notebook is runnable even before you copy `experimental_models.py` into the repository package.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn


@dataclass
class PatchMatchReadoutOutput:
    """One score per candidate graph."""
    scores: torch.Tensor
    raw_similarities: torch.Tensor


class DinoResidualGraphEncoder(nn.Module):
    """
    Preserve the original frozen DINOv2 patch representation and let a graph
    encoder learn only a residual correction.

    graph.x[:, :dino_dim] must contain the original DINOv2 patch token.

        base_encoder: [nodes, input_dim] -> [nodes, base_output_dim]
        delta head:   [nodes, base_output_dim] -> [nodes, dino_dim]
        output:       DINO + beta * delta

    The correction is norm-matched to the original DINO token so beta has a
    stable interpretation. beta=0.05 starts at roughly a 5% correction.
    """

    def __init__(
        self,
        base_encoder: nn.Module,
        base_output_dim: int,
        dino_dim: int = 384,
        initial_patch_residual_scale: float = 0.05,
        norm_match_delta: bool = True,
    ) -> None:
        super().__init__()
        self.base_encoder = base_encoder
        self.base_output_dim = base_output_dim
        self.dino_dim = dino_dim
        self.hidden_dim = dino_dim
        self.norm_match_delta = norm_match_delta

        self.delta_projection = nn.Linear(base_output_dim, dino_dim)
        self.patch_residual_scale = nn.Parameter(
            torch.tensor(float(initial_patch_residual_scale), dtype=torch.float32)
        )

    def forward(self, graph):
        if graph.x.shape[-1] < self.dino_dim:
            raise ValueError(
                f"graph.x has {graph.x.shape[-1]} features, but dino_dim={self.dino_dim}."
            )

        original_dino = graph.x[:, : self.dino_dim].to(torch.float32)
        graph_hidden = self.base_encoder(graph)
        delta = self.delta_projection(graph_hidden)

        if self.norm_match_delta:
            delta = F.normalize(delta, p=2, dim=-1)
            original_norm = (
                original_dino.norm(p=2, dim=-1, keepdim=True)
                .detach()
                .clamp_min(1e-6)
            )
            delta = delta * original_norm

        return original_dino + self.patch_residual_scale * delta


class MaxMeanPatchReadout(nn.Module):
    """
    Patch-to-patch scoring instead of mean-pooling all patches first.

    Default candidate score:

        mean over support images j [
            mean over query patches i [
                max over support patches p cosine(q_i, s_{j,p})
            ]
        ]

    This preserves local correspondences and gives each support image equal
    weight. `classwide_max` instead matches each query patch against all support
    patches of the candidate class jointly.
    """

    def __init__(
        self,
        temperature: float = 0.1,
        support_reduction: str = "mean_image",
    ) -> None:
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        if support_reduction not in {"mean_image", "classwide_max"}:
            raise ValueError(
                "support_reduction must be 'mean_image' or 'classwide_max'."
            )
        self.temperature = float(temperature)
        self.support_reduction = support_reduction

    def _single_graph_score(self, nodes, image_ids):
        query = nodes[image_ids == 0]
        if query.numel() == 0:
            raise ValueError("Candidate graph has no query patches.")
        query = F.normalize(query, p=2, dim=-1)

        support_ids = torch.unique(image_ids[image_ids > 0], sorted=True)
        if support_ids.numel() == 0:
            raise ValueError("Candidate graph has no support patches.")

        if self.support_reduction == "classwide_max":
            support = F.normalize(nodes[image_ids > 0], p=2, dim=-1)
            similarity = query @ support.T
            return similarity.max(dim=1).values.mean()

        per_image_scores = []
        for support_id in support_ids:
            support = F.normalize(nodes[image_ids == support_id], p=2, dim=-1)
            similarity = query @ support.T
            per_image_scores.append(similarity.max(dim=1).values.mean())
        return torch.stack(per_image_scores).mean()

    def forward(self, refined_nodes: torch.Tensor, graph_batch):
        if not hasattr(graph_batch, "image_id"):
            raise AttributeError("graph_batch must contain image_id metadata.")

        image_ids = graph_batch.image_id.reshape(-1).long()
        if hasattr(graph_batch, "batch"):
            graph_ids = graph_batch.batch.reshape(-1).long()
            num_graphs = int(graph_batch.num_graphs)
        else:
            graph_ids = torch.zeros(
                refined_nodes.shape[0], dtype=torch.long, device=refined_nodes.device
            )
            num_graphs = 1

        raw_scores = []
        for graph_id in range(num_graphs):
            mask = graph_ids == graph_id
            raw_scores.append(
                self._single_graph_score(refined_nodes[mask], image_ids[mask])
            )

        raw_scores = torch.stack(raw_scores)
        return PatchMatchReadoutOutput(
            scores=raw_scores / self.temperature,
            raw_similarities=raw_scores,
        )


@torch.inference_mode()
def frozen_patch_match_episode(
    episode: dict,
    device: torch.device,
    temperature: float = 0.1,
    support_reduction: str = "mean_image",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Non-GNN local DINO patch-matching baseline."""
    support = episode["support_patches"].to(device=device, dtype=torch.float32)
    query = episode["query_patches"].to(device=device, dtype=torch.float32)
    labels = episode["query_labels"].to(device=device, dtype=torch.long)

    n_way, k_shot, _, _ = support.shape
    queries_per_class = query.shape[1]
    support = F.normalize(support, p=2, dim=-1)
    query = F.normalize(query, p=2, dim=-1)

    logits_rows = []
    targets = []

    for query_class_position in range(n_way):
        for query_position in range(queries_per_class):
            q = query[query_class_position, query_position]
            candidate_scores = []

            for candidate_id in range(n_way):
                candidate_support = support[candidate_id]

                if support_reduction == "classwide_max":
                    flat_support = candidate_support.reshape(
                        -1, candidate_support.shape[-1]
                    )
                    similarity = q @ flat_support.T
                    raw_score = similarity.max(dim=1).values.mean()
                elif support_reduction == "mean_image":
                    per_image_scores = []
                    for support_index in range(k_shot):
                        similarity = q @ candidate_support[support_index].T
                        per_image_scores.append(
                            similarity.max(dim=1).values.mean()
                        )
                    raw_score = torch.stack(per_image_scores).mean()
                else:
                    raise ValueError(
                        "support_reduction must be 'mean_image' or 'classwide_max'."
                    )

                candidate_scores.append(raw_score / temperature)

            logits_rows.append(torch.stack(candidate_scores))
            targets.append(labels[query_class_position, query_position])

    return torch.stack(logits_rows), torch.stack(targets)


In [ ]:
# Use validation while developing.
EVAL_SPLIT = "val"   # change to "test" only after model decisions are frozen
NUM_EPISODES = 100
QUERIES_PER_CLASS = 15
SEED = 40_000

PROTOCOLS = [
    ("5W5S", 5, 5),
    ("5W1S", 5, 1),
    ("10W5S", 10, 5),
    ("10W1S", 10, 1),
]

config = {
    "experiment_name": "residual_gatv2_dino_patchmatch_5way5shot",
    "input_dim": 387,
    "dino_dim": 384,
    "hidden_dim": 256,
    "num_layers": 2,
    "attention_heads": 4,
    "edge_dim": 5,
    "dropout": 0.1,
    "top_k": 10,
    "min_similarity": None,
    "graph_temperature": 0.1,
    "cls_temperature": 0.1,
    "initial_patch_residual_scale": 0.05,
    "initial_score_residual_scale": 0.05,
    "patch_support_reduction": "mean_image",
    "graph_microbatch_size": 2,
}

restore_feature_splits(
    [EVAL_SPLIT],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    EVAL_SPLIT,
    max_cached_shards=6,
)

print("Evaluation split:", EVAL_SPLIT)
print("Episodes/protocol:", NUM_EPISODES)


## Reconstruct the improved checkpoint

The architecture must match notebook 07 exactly:

`GATv2 387→256 → residual correction 256→384 → original DINO + βΔ → local patch readout → CLS + α·graph`.


In [ ]:
base_gat = PatchGATv2Encoder(
    input_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    heads=config["attention_heads"],
    edge_dim=config["edge_dim"],
    dropout=config["dropout"],
)

encoder = DinoResidualGraphEncoder(
    base_encoder=base_gat,
    base_output_dim=config["hidden_dim"],
    dino_dim=config["dino_dim"],
    initial_patch_residual_scale=config[
        "initial_patch_residual_scale"
    ],
)

readout = MaxMeanPatchReadout(
    temperature=config["graph_temperature"],
    support_reduction=config["patch_support_reduction"],
)

graph_matcher = CrossImageGraphMatcher(
    encoder=encoder,
    readout=readout,
)

model = BaselinePreservingResidualMatcher(
    graph_matcher,
    config["initial_score_residual_scale"],
)

checkpoint_path = (
    paths.drive_checkpoint_dir
    / config["experiment_name"]
    / "best.pt"
)

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Train notebook 07 first. Missing: {checkpoint_path}"
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(features.metadata["grid_size"]),
    top_k=config["top_k"],
    min_similarity=config["min_similarity"],
    graph_dtype=torch.float32,
    similarity_device=device,
)

print("Checkpoint:", checkpoint_path)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print("Best validation accuracy:", checkpoint.get("best_validation_accuracy", "unknown"))
print("Learned score alpha:", float(model.residual_scale.detach().cpu()))
print("Learned patch beta:", float(encoder.patch_residual_scale.detach().cpu()))


In [ ]:
def ci95(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return float("nan")
    return 1.96 * values.std(ddof=1) / math.sqrt(len(values))


@torch.inference_mode()
def evaluate_one_protocol(protocol_name, n_way, k_shot):
    episodes = FewShotFeatureEpisodeDataset(
        features,
        n_way=n_way,
        k_shot=k_shot,
        queries_per_class=QUERIES_PER_CLASS,
        num_episodes=NUM_EPISODES,
        seed=SEED,
        vary_by_epoch=False,
    )

    per_model_episode_acc = {
        "Frozen CLS": [],
        "Frozen mean-patch": [],
        "Frozen local patch match": [],
        "Improved residual GATv2": [],
    }
    per_model_losses = {name: [] for name in per_model_episode_acc}

    fixed_by_model = 0
    broken_by_model = 0

    for episode_index in range(NUM_EPISODES):
        episode = episodes[episode_index]

        cls_logits, cls_targets = frozen_baseline_episode(
            episode,
            "cls",
            device,
            temperature=config["cls_temperature"],
        )

        mean_logits, mean_targets = frozen_baseline_episode(
            episode,
            "mean_patch",
            device,
            temperature=config["graph_temperature"],
        )

        patch_logits, patch_targets = frozen_patch_match_episode(
            episode,
            device=device,
            temperature=config["graph_temperature"],
            support_reduction=config["patch_support_reduction"],
        )

        graph_result = evaluate_residual_feature_episode(
            model,
            graph_builder,
            episode,
            device,
            graph_microbatch_size=config["graph_microbatch_size"],
            cls_temperature=config["cls_temperature"],
        )

        graph_logits = graph_result.logits.to(device)
        graph_targets = graph_result.targets.to(device).long()

        if not (
            torch.equal(cls_targets, mean_targets)
            and torch.equal(cls_targets, patch_targets)
            and torch.equal(cls_targets, graph_targets)
        ):
            raise RuntimeError("Target ordering differs between evaluators.")

        targets = cls_targets

        logits_by_model = {
            "Frozen CLS": cls_logits,
            "Frozen mean-patch": mean_logits,
            "Frozen local patch match": patch_logits,
            "Improved residual GATv2": graph_logits,
        }

        for model_name, logits in logits_by_model.items():
            predictions = logits.argmax(dim=-1)
            accuracy = float(
                predictions.eq(targets).float().mean().item()
            )
            loss = float(F.cross_entropy(logits, targets).item())

            per_model_episode_acc[model_name].append(accuracy)
            per_model_losses[model_name].append(loss)

        cls_correct = cls_logits.argmax(dim=-1).eq(targets)
        graph_correct = graph_logits.argmax(dim=-1).eq(targets)

        fixed_by_model += int((~cls_correct & graph_correct).sum().item())
        broken_by_model += int((cls_correct & ~graph_correct).sum().item())

        if (episode_index + 1) % 10 == 0:
            print(
                f"{protocol_name}: {episode_index + 1}/{NUM_EPISODES} episodes"
            )

    cls_acc = float(np.mean(per_model_episode_acc["Frozen CLS"]))
    rows = []

    for model_name in per_model_episode_acc:
        acc_values = np.asarray(
            per_model_episode_acc[model_name],
            dtype=float,
        )
        accuracy = float(acc_values.mean())

        row = {
            "protocol": protocol_name,
            "model": model_name,
            "loss": float(np.mean(per_model_losses[model_name])),
            "accuracy_percent": 100.0 * accuracy,
            "episode_accuracy_ci95_pp": 100.0 * ci95(acc_values),
            "delta_vs_cls_pp": 100.0 * (accuracy - cls_acc),
        }

        if model_name == "Improved residual GATv2":
            paired_delta = (
                acc_values
                - np.asarray(
                    per_model_episode_acc["Frozen CLS"],
                    dtype=float,
                )
            )
            row.update({
                "paired_gain_ci95_pp": 100.0 * ci95(paired_delta),
                "fixed_by_model": fixed_by_model,
                "broken_by_model": broken_by_model,
                "net_fixed": fixed_by_model - broken_by_model,
                "score_alpha": float(model.residual_scale.detach().cpu()),
                "patch_beta": float(encoder.patch_residual_scale.detach().cpu()),
            })

        rows.append(row)

    return rows


In [ ]:
all_rows = []

for protocol_name, n_way, k_shot in PROTOCOLS:
    print("\n" + "=" * 72)
    print(protocol_name)
    print("=" * 72)
    all_rows.extend(
        evaluate_one_protocol(
            protocol_name,
            n_way,
            k_shot,
        )
    )

summary_df = pd.DataFrame(all_rows)

display(
    summary_df.sort_values(
        ["protocol", "accuracy_percent"],
        ascending=[True, False],
    ).reset_index(drop=True)
)


## How to interpret this experiment

Read the table in this order:

1. **Frozen local patch match vs frozen mean-patch**: if local matching wins, local patch correspondences are useful even without message passing.
2. **Improved residual GATv2 vs frozen CLS**: this is the main end-to-end question.
3. **Improved residual GATv2 vs the old residual GATv2 validation result**: this tests whether preserving DINO patches and changing the readout actually helped.
4. **5W1S and 10W1S**: these have more headroom than 5W5S and are likely more diagnostic.
5. **α and β**: if either collapses close to zero, training is telling you that the corresponding correction is not useful.

Do not tune `top_k`, thresholds, α initialization, β initialization, or readout variants on the test split.
